# 04 Feature Engineering and Train/Test Split

In [8]:
import pandas as pd
import numpy as np
from google.colab import files
uploaded = files.upload()


Saving credit_card_model.csv to credit_card_model (1).csv


In [2]:
df = pd.read_csv("credit_card_model.csv")

print(df.shape)
print(df.columns.tolist())
print(df.isnull().sum().sum(), "missing values")
print()
print(df["default_next_month"].value_counts())
print(df["default_next_month"].value_counts(normalize=True).round(3))

(30000, 24)
['LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_1', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'default_next_month']
0 missing values

default_next_month
0    23364
1     6636
Name: count, dtype: int64
default_next_month
0    0.779
1    0.221
Name: proportion, dtype: float64


In [3]:
pay_cols  = [f"PAY_{i}" for i in range(1, 7)]
bill_cols = [f"BILL_AMT{i}" for i in range(1, 7)]
amt_cols  = [f"PAY_AMT{i}" for i in range(1, 7)]

df["months_late"]     = (df[pay_cols] >= 1).sum(axis=1)
df["zero_pay_months"] = (df[amt_cols] == 0).sum(axis=1)
df["avg_bill"]        = df[bill_cols].mean(axis=1)
df["avg_payment"]     = df[amt_cols].mean(axis=1)
df["utilisation"]     = df["avg_bill"] / df["LIMIT_BAL"]

new_features = ["months_late", "zero_pay_months", "avg_bill", "avg_payment", "utilisation"]

print(df.shape)
print()
print(df.groupby("default_next_month")[new_features].mean().round(3).T)

(30000, 29)

default_next_month          0          1
months_late             0.504      1.996
zero_pay_months         1.080      1.757
avg_bill            45404.818  43470.493
avg_payment          5828.237   3328.216
utilisation             0.351      0.449


In [4]:
print(df[new_features].isnull().sum())
print()
print(df[new_features].describe().round(2).T[["min", "50%", "max"]])
print()
print(df[new_features + ["default_next_month"]].corr()["default_next_month"].round(3))

months_late        0
zero_pay_months    0
avg_bill           0
avg_payment        0
utilisation        0
dtype: int64

                      min       50%        max
months_late          0.00      0.00       6.00
zero_pay_months      0.00      0.00       6.00
avg_bill        -56043.17  21051.83  877313.83
avg_payment          0.00   2397.17  627344.33
utilisation         -0.23      0.28       5.36

months_late           0.398
zero_pay_months       0.163
avg_bill             -0.013
avg_payment          -0.102
utilisation           0.115
default_next_month    1.000
Name: default_next_month, dtype: float64


In [5]:
target = "default_next_month"

X = df.drop(columns=[target])
y = df[target]

print(X.shape)
print(y.shape)
print(X.columns.tolist())

(30000, 28)
(30000,)
['LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_1', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'months_late', 'zero_pay_months', 'avg_bill', 'avg_payment', 'utilisation']


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape, "| Test:", X_test.shape)
print()
print("Defaults in train:", y_train.sum(), "->", round(y_train.mean(), 4))
print("Defaults in test: ", y_test.sum(),  "->", round(y_test.mean(), 4))

Train: (24000, 28) | Test: (6000, 28)

Defaults in train: 5309 -> 0.2212
Defaults in test:  1327 -> 0.2212


In [7]:
X_train.assign(default_next_month=y_train).to_csv("train_set.csv", index=True)
X_test.assign(default_next_month=y_test).to_csv("test_set.csv", index=True)

files.download("train_set.csv")
files.download("test_set.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>